<a href="https://colab.research.google.com/github/mikysetiawan/MachineLearningExpert/blob/master/AlphaZero_TicTacToe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
np.__version__

import math

'2.0.2'

In [2]:
class TicTacToe:
  def __init__(self):
    self.row_count = 3
    self.column_count = 3
    self.action_size = self.row_count * self.column_count

  def get_initial_state(self):
    return np.zeros((self.row_count, self.column_count))

  def get_next_state(self, state, action, player):
    row = action // self.column_count # (Floor Division rounds down), unlike standard division (/) that may resulting decimal, it automatically round
    column = action % self.column_count
    state[row, column] = player
    return state

  def get_valid_moves(self, state):
    return (state.reshape(-1) == 0).astype(np.uint8)

  def check_win(self, state, action):
    row = action // self.column_count
    column = action % self.column_count
    player = state[row, column]

    return (
      np.sum(state[row, :]) == player * self.column_count # check if all column is true
      or np.sum(state[:, column]) == player * self.row_count # check if all row is true
      or np.sum(np.diag(state)) == player * self.row_count # check if all diagonal is true
      or np.sum(np.diag(np.flip(state, axis=0))) == player * self.row_count # check if flip diagonal is true
    )

  def get_value_and_terimanted(self, state, action):
    if self.check_win(state, action):
      return 1, True
    if np.sum(self.get_valid_moves(state)) == 0:
      return 0, True
    return 0, False

  def get_opponent(self, player):
    return -player

In [3]:
tictactoe = TicTacToe()
player = 1

state = tictactoe.get_initial_state()

while True:
  print(state)
  valid_moves = tictactoe.get_valid_moves(state)
  print("valid_moves", [i for i in range(tictactoe.action_size) if valid_moves[i] == 1])

  action = int(input(f"{player}:"))

  if valid_moves[action] == 0:
    print("Invalid move!")
    continue

  state = tictactoe.get_next_state(state, action, player)

  value, is_terminate = tictactoe.get_value_and_terimanted(state, action)
  if is_terminate:
    print(state)
    if value == 1:
      print(player, "won")
    else:
      print("Draw")
    break

  player = tictactoe.get_opponent(player)

[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
valid_moves [0, 1, 2, 3, 4, 5, 6, 7, 8]
1:0
[[1. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
valid_moves [1, 2, 3, 4, 5, 6, 7, 8]
-1:0
Invalid move!
[[1. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
valid_moves [1, 2, 3, 4, 5, 6, 7, 8]
-1:1
[[ 1. -1.  0.]
 [ 0.  0.  0.]
 [ 0.  0.  0.]]
valid_moves [2, 3, 4, 5, 6, 7, 8]
1:5
[[ 1. -1.  0.]
 [ 0.  0.  1.]
 [ 0.  0.  0.]]
valid_moves [2, 3, 4, 6, 7, 8]
-1:4
[[ 1. -1.  0.]
 [ 0. -1.  1.]
 [ 0.  0.  0.]]
valid_moves [2, 3, 6, 7, 8]
1:3
[[ 1. -1.  0.]
 [ 1. -1.  1.]
 [ 0.  0.  0.]]
valid_moves [2, 6, 7, 8]
-1:8
[[ 1. -1.  0.]
 [ 1. -1.  1.]
 [ 0.  0. -1.]]
valid_moves [2, 6, 7]
1:6
[[ 1. -1.  0.]
 [ 1. -1.  1.]
 [ 1.  0. -1.]]
1 won


In [ ]:
class Node:
  def __init__(self, game, args, state, parent=None, action_taken=None):
    self.state = state
    self.game = game
    self.args = args
    self.parent = parent
    self.action_taken = action_taken

    self.children = []
    self.expandable_moves = game.get_valid_moves(state)

    self.visit_count = 0
    self.value_sum = 0

  def is_fully_expanded(self):
    return np.sum(self.expandable_moves) == 0 and len(self.children) > 0

  def select(self):
    best_child = None
    best_ucb = -np.inf

    for child in self.children:
      ucb = self.get_ucb(child)
      if ucb > best_ucb:
        best_child = child
        best_ucb = ucb
    return best_child

  def get_ucb(self, child):
    q_value = 1 - ((child.value_sum / child.visit_count) + 1) / 2
    # + 1) / 2: to shifts and scales the values from [-1, 1] to a clean [0, 1] range (where 0 is a total loss and 1 is a perfect win)
    # "1 - (...): This inverts the score. Because games alternate turns (gantian), a position that is excellent for the opponent (the child) is terrible for the current player (the parent). Inverting it ensures the parent selects a move that is good for themselves, not the opponent.
    return q_value + self.args["C"] * math.sqrt(math.log(self.visit_count) / child.visit_count)

class MCTS:
  def __init__(self, game, args):
    self.game = game
    self.args = args

  def search(self, state):
    # define root
    root = Node(self.game, self.args, state)

    for search in range(self.args["num_searches"]):
      mode = root

      while node.is_fully_expanded():
        node = node.select()
      # selection
      # expansion
      # simulation
      # backpropagation

    # return visit_counts